# Using PRQL to query data



- [Query language: wikipedia]( https://en.wikipedia.org/wiki/Query_language )
- [PRQL - Pipelined Relational Query Language, pronounced “Prequel”
]( https://prql-lang.org/ )
- [PRQL in DuckDB]( https://duckdb.org/community_extensions/extensions/prql )


In [1]:
%%capture pip_output
%%bash
pip install pyprql


In [2]:
import duckdb
import prqlc
import pandas as pd
import pyprql.pandas_accessor


In [3]:
data = {
    'region': ['West'] * 20,
    'state': ['NM']*10 + ['AZ']*10,
    'county': ['Bernalillo']*5 + ['Santa Fe']*5 + ['Maricopa']*5 + ['Pima']*5,
    'city': [
        'Albuquerque', 'Albuquerque', 'Rio Rancho', 'Tijeras', 'Cedar Crest',
        'Santa Fe', 'Santa Fe', 'Espanola', 'Edgewood', 'Glorieta',
        'Phoenix', 'Mesa', 'Chandler', 'Scottsdale', 'Tempe',
        'Tucson', 'Marana', 'Oro Valley', 'Sahuarita', 'South Tucson'
    ],
    'tract': [f'T-{i:03d}' for i in range(1, 21)],
    'block': [f'B-{i:04d}' for i in range(101, 121)],
    'population': [2500, 1800, 3100, 450, 600, 2100, 1900, 800, 1200, 300,
                   5000, 4200, 3800, 3100, 2900, 4500, 1100, 950, 700, 400]
}
# data


In [4]:
df = pd.DataFrame(data)
df


,region,state,county,city,tract,block,population
0,West,NM,Bernalillo,Albuquerque,T-001,B-0101,2500
1,West,NM,Bernalillo,Albuquerque,T-002,B-0102,1800
2,West,NM,Bernalillo,Rio Rancho,T-003,B-0103,3100
3,West,NM,Bernalillo,Tijeras,T-004,B-0104,450
4,West,NM,Bernalillo,Cedar Crest,T-005,B-0105,600
5,West,NM,Santa Fe,Santa Fe,T-006,B-0106,2100
6,West,NM,Santa Fe,Santa Fe,T-007,B-0107,1900
7,West,NM,Santa Fe,Espanola,T-008,B-0108,800
8,West,NM,Santa Fe,Edgewood,T-009,B-0109,1200
9,West,NM,Santa Fe,Glorieta,T-010,B-0110,300


## Using Pipelined Relational Query Language ( PRQL )




### Example 0


In [5]:
prql_query = '''
  from df
'''


In [6]:
sql_query = prqlc.compile(prql_query)
print(sql_query)


SELECT
  *
FROM
  df

-- Generated by PRQL compiler version:0.13.14 (https://prql-lang.org)



In [9]:
duckdb.sql(sql_query)


┌─────────┬─────────┬────────────┬──────────────┬─────────┬─────────┬────────────┐
│ region  │  state  │   county   │     city     │  tract  │  block  │ population │
│ varchar │ varchar │  varchar   │   varchar    │ varchar │ varchar │   int64    │
├─────────┼─────────┼────────────┼──────────────┼─────────┼─────────┼────────────┤
│ West    │ NM      │ Bernalillo │ Albuquerque  │ T-001   │ B-0101  │       2500 │
│ West    │ NM      │ Bernalillo │ Albuquerque  │ T-002   │ B-0102  │       1800 │
│ West    │ NM      │ Bernalillo │ Rio Rancho   │ T-003   │ B-0103  │       3100 │
│ West    │ NM      │ Bernalillo │ Tijeras      │ T-004   │ B-0104  │        450 │
│ West    │ NM      │ Bernalillo │ Cedar Crest  │ T-005   │ B-0105  │        600 │
│ West    │ NM      │ Santa Fe   │ Santa Fe     │ T-006   │ B-0106  │       2100 │
│ West    │ NM      │ Santa Fe   │ Santa Fe     │ T-007   │ B-0107  │       1900 │
│ West    │ NM      │ Santa Fe   │ Espanola     │ T-008   │ B-0108  │        800 │
│ We

### Example 1


In [10]:
# Using pandas
df[['city', 'county', 'population']]


,city,county,population
0,Albuquerque,Bernalillo,2500
1,Albuquerque,Bernalillo,1800
2,Rio Rancho,Bernalillo,3100
3,Tijeras,Bernalillo,450
4,Cedar Crest,Bernalillo,600
5,Santa Fe,Santa Fe,2100
6,Santa Fe,Santa Fe,1900
7,Espanola,Santa Fe,800
8,Edgewood,Santa Fe,1200
9,Glorieta,Santa Fe,300


In [11]:
# using PRQL
query = '''
  select { city, county, population }
'''

df.prql.query( query )


,city,county,population
0,Albuquerque,Bernalillo,2500
1,Albuquerque,Bernalillo,1800
2,Rio Rancho,Bernalillo,3100
3,Tijeras,Bernalillo,450
4,Cedar Crest,Bernalillo,600
5,Santa Fe,Santa Fe,2100
6,Santa Fe,Santa Fe,1900
7,Espanola,Santa Fe,800
8,Edgewood,Santa Fe,1200
9,Glorieta,Santa Fe,300


In [12]:
# Using PRQL and DuckDB
sql_query = prqlc.compile('''
  from df
  select { city, county, population }
''')
duckdb.sql(sql_query)


┌──────────────┬────────────┬────────────┐
│     city     │   county   │ population │
│   varchar    │  varchar   │   int64    │
├──────────────┼────────────┼────────────┤
│ Albuquerque  │ Bernalillo │       2500 │
│ Albuquerque  │ Bernalillo │       1800 │
│ Rio Rancho   │ Bernalillo │       3100 │
│ Tijeras      │ Bernalillo │        450 │
│ Cedar Crest  │ Bernalillo │        600 │
│ Santa Fe     │ Santa Fe   │       2100 │
│ Santa Fe     │ Santa Fe   │       1900 │
│ Espanola     │ Santa Fe   │        800 │
│ Edgewood     │ Santa Fe   │       1200 │
│ Glorieta     │ Santa Fe   │        300 │
│ Phoenix      │ Maricopa   │       5000 │
│ Mesa         │ Maricopa   │       4200 │
│ Chandler     │ Maricopa   │       3800 │
│ Scottsdale   │ Maricopa   │       3100 │
│ Tempe        │ Maricopa   │       2900 │
│ Tucson       │ Pima       │       4500 │
│ Marana       │ Pima       │       1100 │
│ Oro Valley   │ Pima       │        950 │
│ Sahuarita    │ Pima       │        700 │
│ South Tuc

### Example 2



In [13]:
# Using pandas
df[['city', 'county', 'population']].head(3)


,city,county,population
0,Albuquerque,Bernalillo,2500
1,Albuquerque,Bernalillo,1800
2,Rio Rancho,Bernalillo,3100


In [14]:
# using PRQL
query = '''
  select { city, county, population }
  take 3
'''

df.prql.query( query )



,city,county,population
0,Albuquerque,Bernalillo,2500
1,Albuquerque,Bernalillo,1800
2,Rio Rancho,Bernalillo,3100


In [15]:
# Using PRQL and DuckDB
sql_query = prqlc.compile('''
  from df
  select { city, county, population }
  take 3
''')
duckdb.sql(sql_query)


┌─────────────┬────────────┬────────────┐
│    city     │   county   │ population │
│   varchar   │  varchar   │   int64    │
├─────────────┼────────────┼────────────┤
│ Albuquerque │ Bernalillo │       2500 │
│ Albuquerque │ Bernalillo │       1800 │
│ Rio Rancho  │ Bernalillo │       3100 │
└─────────────┴────────────┴────────────┘

### Example 3



In [16]:
# using pandas
(
df
  [['city', 'county', 'population']]
  [ df['population'] > 2000 ]
  .sort_values('population', ascending=False)
  .head(3)
)

,city,county,population
10,Phoenix,Maricopa,5000
15,Tucson,Pima,4500
11,Mesa,Maricopa,4200


In [17]:
# using PRQL
query = '''
  select { city, county, population }
  filter ( population > 2000 )
  sort { -population }
  take 3
'''

df.prql.query( query )


,city,county,population
0,Phoenix,Maricopa,5000
1,Tucson,Pima,4500
2,Mesa,Maricopa,4200


In [18]:
# Using PRQL and DuckDB
sql_query = prqlc.compile('''
  from df
  select { city, county, population }
  filter ( population > 2000 )
  sort { -population }
  take 3
''')
duckdb.sql(sql_query)


┌─────────┬──────────┬────────────┐
│  city   │  county  │ population │
│ varchar │ varchar  │   int64    │
├─────────┼──────────┼────────────┤
│ Phoenix │ Maricopa │       5000 │
│ Tucson  │ Pima     │       4500 │
│ Mesa    │ Maricopa │       4200 │
└─────────┴──────────┴────────────┘

In [27]:
# Using PRQL and DuckDB -- reordered
sql_query = prqlc.compile('''
  from df                               # the data frame
  # filter ( population > 2000 )          # filter the rows
  # select { city, county, population }   # select the columns
  # sort { -population }                  # sort, descending
  # take 3                                # filter the first 3
''')
print(sql_query)
duckdb.sql(sql_query)


SELECT
  *
FROM
  df
ORDER BY
  population DESC

-- Generated by PRQL compiler version:0.13.14 (https://prql-lang.org)



┌─────────┬─────────┬────────────┬──────────────┬─────────┬─────────┬────────────┐
│ region  │  state  │   county   │     city     │  tract  │  block  │ population │
│ varchar │ varchar │  varchar   │   varchar    │ varchar │ varchar │   int64    │
├─────────┼─────────┼────────────┼──────────────┼─────────┼─────────┼────────────┤
│ West    │ AZ      │ Maricopa   │ Phoenix      │ T-011   │ B-0111  │       5000 │
│ West    │ AZ      │ Pima       │ Tucson       │ T-016   │ B-0116  │       4500 │
│ West    │ AZ      │ Maricopa   │ Mesa         │ T-012   │ B-0112  │       4200 │
│ West    │ AZ      │ Maricopa   │ Chandler     │ T-013   │ B-0113  │       3800 │
│ West    │ NM      │ Bernalillo │ Rio Rancho   │ T-003   │ B-0103  │       3100 │
│ West    │ AZ      │ Maricopa   │ Scottsdale   │ T-014   │ B-0114  │       3100 │
│ West    │ AZ      │ Maricopa   │ Tempe        │ T-015   │ B-0115  │       2900 │
│ West    │ NM      │ Bernalillo │ Albuquerque  │ T-001   │ B-0101  │       2500 │
│ We